# Update Gas ATB Capex with Halcyon Regression Results

After our regression analysis, we build a new version of `inputs/plant_characteristics/gas_ATB_2024_moderate.csv`, replacing `capcost` for **Gas-CC** and **Gas-CT** in years **2026-2032** with the forecasts produced by `CCGT_gas_capex.ipynb` and `CT_gas_capex.ipynb`. All other technologies (`Gas-CC_H_1x1`, `Gas-CC_H_2x1`, `Gas-CT_aero`) and all other years are left untouched. This is saved as a new file, without modifying the original `gas_ATB_2024_moderate.csv`.

This notebook uses the forecast CSVs already exported by the two source notebooks (`ccgt_regression_forecast.csv`, `ct_regression_forecast.csv`). Run those notebooks first if you want to refresh the forecasts.

We now produce a **single** scenario (`gas-ccgt_CEPM_all.csv`) — Gas-CC and Gas-CT both come from a single regression on all the data, no clustering. We previously built 3 versions (low/high/all cost tiers for CCGT), but per `CCGT_clustering_methods.ipynb`, no clustering method gives a robust, meaningful low/high split for CCGT, so we dropped that approach in favor of one all-data case.

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "runreeds.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

PLANTCHAR_DIR = REPO_ROOT / "inputs/plant_characteristics"
FORECAST_DIR = REPO_ROOT / "CEPM/preprocessing/gas_capex_forecast"

ATB_PATH = PLANTCHAR_DIR / "gas_ATB_2024_moderate.csv"

ccgt_forecast = pd.read_csv(FORECAST_DIR / "ccgt_regression_forecast.csv").set_index("Year")["Cost_$/kW"]
ct_forecast = pd.read_csv(FORECAST_DIR / "ct_regression_forecast.csv").set_index("Year")["Cost_$/kW"]
years = ccgt_forecast.index

ccgt_forecast

Year
2026    1327.4
2027    1469.8
2028    1612.2
2029    1754.5
2030    1896.9
2031    2039.3
2032    2181.7
Name: Cost_$/kW, dtype: float64

## Build the ATB update

In [2]:
atb_base = pd.read_csv(ATB_PATH)
atb = atb_base.copy()

is_cc_forecast_years = (atb["i"] == "Gas-CC") & (atb["t"].isin(years))
atb.loc[is_cc_forecast_years, "capcost"] = atb.loc[is_cc_forecast_years, "t"].map(ccgt_forecast).values

is_ct_forecast_years = (atb["i"] == "Gas-CT") & (atb["t"].isin(years))
atb.loc[is_ct_forecast_years, "capcost"] = atb.loc[is_ct_forecast_years, "t"].map(ct_forecast).values

out_path = PLANTCHAR_DIR / "gas-ccgt_CEPM_all.csv"
atb.to_csv(out_path, index=False)

atb[(atb["i"].isin(["Gas-CC", "Gas-CT"])) & (atb["t"].isin(years))]

,i,t,capcost,fom,vom,heatrate
16,Gas-CC,2026,1327.4,33.1,2.10,6.300
17,Gas-CC,2027,1469.8,32.8,2.09,6.285
18,Gas-CC,2028,1612.2,32.4,2.07,6.269
19,Gas-CC,2029,1754.5,32.1,2.06,6.253
20,Gas-CC,2030,1896.9,31.8,2.04,6.238
21,Gas-CC,2031,2039.3,31.4,2.02,6.222
22,Gas-CC,2032,2181.7,31.1,2.01,6.206
57,Gas-CT,2026,1163.7,25.6,6.94,9.717
58,Gas-CT,2027,1258.5,25.4,6.94,9.717
59,Gas-CT,2028,1353.2,25.3,6.94,9.717


## Comparison with ATB moderate

In [3]:
compare = atb_base[(atb_base["i"].isin(["Gas-CC", "Gas-CT"])) & (atb_base["t"].isin(years))][["i", "t", "capcost"]]
compare = compare.rename(columns={"capcost": "ATB_original"})
compare = compare.merge(
    atb[(atb["i"].isin(["Gas-CC", "Gas-CT"])) & (atb["t"].isin(years))][["i", "t", "capcost"]].rename(columns={"capcost": "CEPM_all"}),
    on=["i", "t"],
)
compare.sort_values(["i", "t"]).reset_index(drop=True)

,i,t,ATB_original,CEPM_all
0,Gas-CC,2026,1202.4,1327.4
1,Gas-CC,2027,1191.6,1469.8
2,Gas-CC,2028,1180.8,1612.2
3,Gas-CC,2029,1170.1,1754.5
4,Gas-CC,2030,1159.3,1896.9
5,Gas-CC,2031,1148.5,2039.3
6,Gas-CC,2032,1137.8,2181.7
7,Gas-CT,2026,1075.2,1163.7
8,Gas-CT,2027,1066.4,1258.5
9,Gas-CT,2028,1057.6,1353.2
